In [1]:
# SPARC FROZEN VALIDATION REPRODUCTION — CELL 1
# Setup, package lock, input integrity check, and locked-data loading.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys

RELEASE_ROOT = Path(
    "/content/drive/MyDrive/UQSH_DarkMatter_Project/"
    "reproducibility_exports/"
    "sparc-clean-independent-reconstruction-20260812_124457_UTC/"
    "SPARC_Clean_Independent_Reconstruction"
)

LOCK_FILE = RELEASE_ROOT / "requirements-lock.txt"
MANIFEST_FILE = RELEASE_ROOT / "stage1_manifest.json"

if not RELEASE_ROOT.is_dir():
    raise FileNotFoundError(f"Release folder not found:\n{RELEASE_ROOT}")

if not LOCK_FILE.is_file():
    raise FileNotFoundError(f"Package lock not found:\n{LOCK_FILE}")

if not MANIFEST_FILE.is_file():
    raise FileNotFoundError(f"Manifest not found:\n{MANIFEST_FILE}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(LOCK_FILE)],
    check=True,
)

import numpy as np
import pandas as pd

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

manifest = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
integrity_problems = []

for entry in manifest["copied_files"]:
    relative_path = entry["release_relative_path"]
    expected_hash = entry["copied_sha256"]
    path = RELEASE_ROOT / relative_path

    if not path.is_file():
        integrity_problems.append(f"MISSING: {relative_path}")
    elif sha256_file(path) != expected_hash:
        integrity_problems.append(f"HASH MISMATCH: {relative_path}")

if integrity_problems:
    print("=== INPUT INTEGRITY: FAIL ===")

    for problem in integrity_problems:
        print("-", problem)

    raise RuntimeError("The locked reproduction inputs are not intact.")

INPUT_DIR = RELEASE_ROOT / "inputs"
REFERENCE_DIR = RELEASE_ROOT / "locked_reference" / "kernel_variant_nested_validation"

MASTER_FILE = (
    REFERENCE_DIR / "tables" / "kernel_variant_master_table.csv"
)

FOLD_RESULTS_FILE = (
    REFERENCE_DIR / "tables" / "kernel_variant_nested_fold_results.csv"
)

PERFORMANCE_FILE = (
    REFERENCE_DIR / "tables" / "kernel_variant_performance_summary.csv"
)

required_files = [
    MASTER_FILE,
    FOLD_RESULTS_FILE,
    PERFORMANCE_FILE,
]

for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(f"Required locked file missing:\n{path}")

master = pd.read_csv(MASTER_FILE, low_memory=False)
locked_folds = pd.read_csv(FOLD_RESULTS_FILE, low_memory=False)
locked_performance = pd.read_csv(PERFORMANCE_FILE, low_memory=False)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
RUN_DIR = RELEASE_ROOT / "public_colab_runs" / f"frozen_validation_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=False)

print("=== CELL 1 COMPLETE ===")
print("Input integrity: PASS")
print("Master table:", master.shape)
print("Frozen fold table:", locked_folds.shape)
print("Performance reference:", locked_performance.shape)
print("New output folder:", RUN_DIR)

Mounted at /content/drive
=== CELL 1 COMPLETE ===
Input integrity: PASS
Master table: (7560, 205)
Frozen fold table: (120, 20)
Performance reference: (24, 13)
New output folder: /content/drive/MyDrive/UQSH_DarkMatter_Project/reproducibility_exports/sparc-clean-independent-reconstruction-20260812_124457_UTC/SPARC_Clean_Independent_Reconstruction/public_colab_runs/frozen_validation_20260813_080154_UTC


In [2]:
# SPARC FROZEN VALIDATION REPRODUCTION — CELL 2
# Exact re-execution of all 60 frozen outer-fold model fits.

from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

TARGET_COLUMN = "log_g_field_obs"

COMMON_FEATURES = [
    "log_r_over_Rdisk",
    "log_local_SBdisk",
]

required_columns = [
    "Galaxy",
    "sample_analysis",
    "outer_fold",
    TARGET_COLUMN,
] + COMMON_FEATURES

missing_columns = [
    column for column in required_columns
    if column not in master.columns
]

if missing_columns:
    raise KeyError(
        "Required master-table columns missing:\n"
        + "\n".join(missing_columns)
    )

frozen_folds = locked_folds[
    locked_folds["selection_mode"] == "frozen_reference"
].copy()

if len(frozen_folds) != 60:
    raise RuntimeError(
        f"Expected 60 frozen fold specifications, found {len(frozen_folds)}."
    )

def parse_global_features(value):
    if pd.isna(value):
        return []

    text = str(value).strip()

    if text == "" or text.lower() in {"none", "nan"}:
        return []

    return [
        item.strip()
        for item in text.split(",")
        if item.strip()
    ]

oof_tables = []
fold_check_rows = []

for _, setting in frozen_folds.iterrows():
    sample_name = str(setting["sample"])
    kernel_variant = str(setting["kernel_variant"])
    outer_fold = int(setting["outer_fold"])
    kernel_feature = str(setting["selected_kernel_feature"])
    global_features = parse_global_features(
        setting["selected_global_features"]
    )
    ridge_alpha = float(setting["selected_ridge_alpha"])

    feature_columns = (
        [kernel_feature] +
        COMMON_FEATURES +
        global_features
    )

    missing_features = [
        column for column in feature_columns
        if column not in master.columns
    ]

    if missing_features:
        raise KeyError(
            f"Features missing for {sample_name}, {kernel_variant}, "
            f"fold {outer_fold}:\n"
            + "\n".join(missing_features)
        )

    use = master[
        master["sample_analysis"].astype(str) == sample_name
    ].copy()

    use = use[
        [
            "Galaxy",
            "sample_analysis",
            "outer_fold",
            "g_bar_m_s2",
            "log_g_obs",
            TARGET_COLUMN,
        ] + feature_columns
    ].replace([np.inf, -np.inf], np.nan).dropna().copy()

    train = use[use["outer_fold"] != outer_fold].copy()
    test = use[use["outer_fold"] == outer_fold].copy()

    if len(train) != int(setting["n_train_rows"]):
        raise RuntimeError(
            f"Training-row mismatch for {sample_name}, {kernel_variant}, "
            f"fold {outer_fold}: {len(train)} vs "
            f"{int(setting['n_train_rows'])}"
        )

    if len(test) != int(setting["n_test_rows"]):
        raise RuntimeError(
            f"Test-row mismatch for {sample_name}, {kernel_variant}, "
            f"fold {outer_fold}: {len(test)} vs "
            f"{int(setting['n_test_rows'])}"
        )

    model = Pipeline([
        ("scale", StandardScaler()),
        ("ridge", Ridge(alpha=ridge_alpha)),
    ])

    model.fit(
        train[feature_columns].to_numpy(),
        train[TARGET_COLUMN].to_numpy(),
    )

    prediction = model.predict(
        test[feature_columns].to_numpy()
    )

    output = test[
        [
            "Galaxy",
            "sample_analysis",
            "outer_fold",
            "g_bar_m_s2",
            "log_g_obs",
            TARGET_COLUMN,
        ]
    ].copy()

    output = output.rename(
        columns={"sample_analysis": "sample"}
    )

    output["kernel_variant"] = kernel_variant
    output["selected_kernel_feature"] = kernel_feature
    output["selected_global_features"] = ",".join(global_features)
    output["selected_ridge_alpha"] = ridge_alpha
    output["prediction_frozen_design_oof"] = prediction
    output["residual_frozen_design_oof"] = (
        test[TARGET_COLUMN].to_numpy() - prediction
    )

    oof_tables.append(output)

    fold_check_rows.append({
        "sample": sample_name,
        "kernel_variant": kernel_variant,
        "outer_fold": outer_fold,
        "selected_kernel_feature": kernel_feature,
        "selected_global_features": ",".join(global_features),
        "selected_ridge_alpha": ridge_alpha,
        "n_train_rows": len(train),
        "n_test_rows": len(test),
        "reconstructed_test_rmse": float(
            mean_squared_error(
                test[TARGET_COLUMN].to_numpy(),
                prediction,
            ) ** 0.5
        ),
    })

exact_oof = pd.concat(oof_tables, ignore_index=True)
exact_fold_check = pd.DataFrame(fold_check_rows)

field_summary_rows = []

for (sample_name, kernel_variant), group in exact_oof.groupby(
    ["sample", "kernel_variant"],
    sort=True,
):
    y_true = group[TARGET_COLUMN].to_numpy()
    y_pred = group["prediction_frozen_design_oof"].to_numpy()

    field_summary_rows.append({
        "sample": sample_name,
        "kernel_variant": kernel_variant,
        "n_rows": int(len(group)),
        "n_galaxies": int(group["Galaxy"].nunique()),
        "reconstructed_field_rmse": float(
            mean_squared_error(y_true, y_pred) ** 0.5
        ),
        "reconstructed_field_r2": float(
            r2_score(y_true, y_pred)
        ),
    })

field_summary = pd.DataFrame(field_summary_rows)

locked_field = locked_performance[
    locked_performance["selection_mode"] == "frozen_reference"
][
    [
        "sample",
        "kernel_variant",
        "n_rows",
        "n_galaxies",
        "field_rmse",
        "field_r2",
    ]
].copy()

field_comparison = field_summary.merge(
    locked_field,
    on=["sample", "kernel_variant"],
    how="left",
    suffixes=("_reconstructed", "_locked"),
)

field_comparison["rmse_difference"] = (
    field_comparison["reconstructed_field_rmse"] -
    field_comparison["field_rmse"]
)

field_comparison["r2_difference"] = (
    field_comparison["reconstructed_field_r2"] -
    field_comparison["field_r2"]
)

field_comparison["field_metrics_exact"] = (
    field_comparison["rmse_difference"].abs() < 1e-10
) & (
    field_comparison["r2_difference"].abs() < 1e-10
)

exact_oof.to_csv(
    RUN_DIR / "frozen_design_oof_predictions.csv",
    index=False,
)

exact_fold_check.to_csv(
    RUN_DIR / "frozen_design_fold_check.csv",
    index=False,
)

field_comparison.to_csv(
    RUN_DIR / "frozen_design_field_vs_locked_comparison.csv",
    index=False,
)

all_field_exact = bool(
    field_comparison["field_metrics_exact"].all()
)

print("=== CELL 2 COMPLETE ===")
print("Reconstructed OOF rows:", len(exact_oof))
print("Reconstructed fold fits:", len(exact_fold_check))
print("All field metrics exact:", all_field_exact)
print()

display(
    field_comparison.sort_values(
        ["sample", "kernel_variant"]
    ).reset_index(drop=True)
)

=== CELL 2 COMPLETE ===
Reconstructed OOF rows: 30240
Reconstructed fold fits: 60
All field metrics exact: True



,sample,kernel_variant,n_rows_reconstructed,n_galaxies_reconstructed,reconstructed_field_rmse,reconstructed_field_r2,n_rows_locked,n_galaxies_locked,field_rmse,field_r2,rmse_difference,r2_difference,field_metrics_exact
0,Q1,abs_leave_one_out,1859,99,0.316181,0.383767,1859,99,0.316181,0.383767,0.000000e+00,0.000000e+00,True
1,Q1,abs_self_dlnr,1859,99,0.314117,0.391784,1859,99,0.314117,0.391784,5.551115e-17,5.551115e-17,True
2,Q1,current_abs_self_uniform,1859,99,0.313284,0.395006,1859,99,0.313284,0.395006,0.000000e+00,0.000000e+00,True
3,Q1,signed_self_uniform,1859,99,0.313228,0.395225,1859,99,0.313228,0.395225,5.551115e-17,5.551115e-17,True
4,Q12,abs_leave_one_out,2805,163,0.336814,0.421937,2805,163,0.336814,0.421937,5.551115e-17,0.000000e+00,True
5,Q12,abs_self_dlnr,2805,163,0.335313,0.427077,2805,163,0.335313,0.427077,5.551115e-17,0.000000e+00,True
6,Q12,current_abs_self_uniform,2805,163,0.335103,0.427794,2805,163,0.335103,0.427794,5.551115e-17,5.551115e-17,True
7,Q12,signed_self_uniform,2805,163,0.335055,0.427956,2805,163,0.335055,0.427956,5.551115e-17,0.000000e+00,True
8,Q123,abs_leave_one_out,2896,175,0.359918,0.407846,2896,175,0.359918,0.407846,5.551115e-17,0.000000e+00,True
9,Q123,abs_self_dlnr,2896,175,0.358593,0.412197,2896,175,0.358593,0.412197,0.000000e+00,0.000000e+00,True


In [3]:
# SPARC FROZEN VALIDATION REPRODUCTION — CELL 3
# Exact reconstruction check for the derived observed acceleration g_obs.

from sklearn.metrics import mean_squared_error, r2_score

if "exact_oof" not in globals():
    raise RuntimeError("Run CELL 2 first.")

required_columns = [
    "g_bar_m_s2",
    "log_g_obs",
    "prediction_frozen_design_oof",
]

missing_columns = [
    column for column in required_columns
    if column not in exact_oof.columns
]

if missing_columns:
    raise KeyError(
        "Required OOF columns missing:\n"
        + "\n".join(missing_columns)
    )

gobs_check = exact_oof.copy()

gobs_check["g_field_pred_m_s2"] = np.power(
    10.0,
    gobs_check["prediction_frozen_design_oof"]
)

gobs_check["g_obs_pred_m_s2"] = (
    gobs_check["g_bar_m_s2"] +
    gobs_check["g_field_pred_m_s2"]
)

gobs_check["log_g_obs_pred"] = np.log10(
    gobs_check["g_obs_pred_m_s2"]
)

gobs_summary_rows = []

for (sample_name, kernel_variant), group in gobs_check.groupby(
    ["sample", "kernel_variant"],
    sort=True,
):
    y_true = group["log_g_obs"].to_numpy()
    y_pred = group["log_g_obs_pred"].to_numpy()

    gobs_summary_rows.append({
        "sample": sample_name,
        "kernel_variant": kernel_variant,
        "n_rows": int(len(group)),
        "reconstructed_gobs_rmse": float(
            mean_squared_error(y_true, y_pred) ** 0.5
        ),
        "reconstructed_gobs_r2": float(
            r2_score(y_true, y_pred)
        ),
    })

gobs_summary = pd.DataFrame(gobs_summary_rows)

locked_gobs = locked_performance[
    locked_performance["selection_mode"] == "frozen_reference"
][
    [
        "sample",
        "kernel_variant",
        "gobs_rmse",
        "gobs_r2",
    ]
].copy()

gobs_comparison = gobs_summary.merge(
    locked_gobs,
    on=["sample", "kernel_variant"],
    how="left",
)

gobs_comparison["gobs_rmse_difference"] = (
    gobs_comparison["reconstructed_gobs_rmse"] -
    gobs_comparison["gobs_rmse"]
)

gobs_comparison["gobs_r2_difference"] = (
    gobs_comparison["reconstructed_gobs_r2"] -
    gobs_comparison["gobs_r2"]
)

gobs_comparison["gobs_metrics_exact"] = (
    gobs_comparison["gobs_rmse_difference"].abs() < 1e-10
) & (
    gobs_comparison["gobs_r2_difference"].abs() < 1e-10
)

gobs_check.to_csv(
    RUN_DIR / "frozen_design_gobs_oof_predictions.csv",
    index=False,
)

gobs_comparison.to_csv(
    RUN_DIR / "frozen_design_gobs_vs_locked_comparison.csv",
    index=False,
)

all_gobs_exact = bool(
    gobs_comparison["gobs_metrics_exact"].all()
)

print("=== CELL 3 COMPLETE ===")
print("Reconstructed rows:", len(gobs_check))
print("All g_obs metrics exact:", all_gobs_exact)
print()

display(
    gobs_comparison.sort_values(
        ["sample", "kernel_variant"]
    ).reset_index(drop=True)
)

=== CELL 3 COMPLETE ===
Reconstructed rows: 30240
All g_obs metrics exact: True



,sample,kernel_variant,n_rows,reconstructed_gobs_rmse,reconstructed_gobs_r2,gobs_rmse,gobs_r2,gobs_rmse_difference,gobs_r2_difference,gobs_metrics_exact
0,Q1,abs_leave_one_out,1859,0.147835,0.904134,0.147835,0.904134,2.775558e-17,0.000000e+00,True
1,Q1,abs_self_dlnr,1859,0.145679,0.906911,0.145679,0.906911,8.326673e-17,0.000000e+00,True
2,Q1,current_abs_self_uniform,1859,0.145441,0.907215,0.145441,0.907215,5.551115e-17,-2.220446e-16,True
3,Q1,signed_self_uniform,1859,0.145289,0.907407,0.145289,0.907407,5.551115e-17,0.000000e+00,True
4,Q12,abs_leave_one_out,2805,0.163695,0.903169,0.163695,0.903169,5.551115e-17,-2.220446e-16,True
5,Q12,abs_self_dlnr,2805,0.161697,0.905518,0.161697,0.905518,0.000000e+00,0.000000e+00,True
6,Q12,current_abs_self_uniform,2805,0.161875,0.905310,0.161875,0.905310,5.551115e-17,0.000000e+00,True
7,Q12,signed_self_uniform,2805,0.161863,0.905325,0.161863,0.905325,0.000000e+00,0.000000e+00,True
8,Q123,abs_leave_one_out,2896,0.174639,0.893003,0.174639,0.893003,5.551115e-17,0.000000e+00,True
9,Q123,abs_self_dlnr,2896,0.172467,0.895648,0.172467,0.895648,5.551115e-17,0.000000e+00,True


In [4]:
# SPARC FROZEN VALIDATION REPRODUCTION — CELL 4
# Writes the English validation certificate and checksums for this run.

from importlib.metadata import version
import platform

if "field_comparison" not in globals() or "gobs_comparison" not in globals():
    raise RuntimeError("Run CELL 2 and CELL 3 first.")

field_pass = bool(field_comparison["field_metrics_exact"].all())
gobs_pass = bool(gobs_comparison["gobs_metrics_exact"].all())
overall_pass = bool(field_pass and gobs_pass)

certificate_lines = [
    "# SPARC frozen validation reproduction certificate",
    "",
    "## Result",
    "",
    f"Overall validation status: {'PASS' if overall_pass else 'FAIL'}",
    f"Field-response metrics reproduced exactly: {field_pass}",
    f"Reconstructed observed-acceleration metrics reproduced exactly: {gobs_pass}",
    "",
    "## Scope",
    "",
    "This notebook re-executes the frozen SPARC kernel-variant nested-validation",
    "design from archived derived inputs, archived kernel features, frozen outer",
    "fold assignments, frozen model specifications, and frozen Ridge parameters.",
    "",
    "The official SPARC raw archive is not redistributed by this release.",
    "",
    "## Numerical criterion",
    "",
    "A metric is treated as exact when the absolute difference from the locked",
    "reference is below 1e-10. Observed residual differences of order 1e-16 are",
    "standard floating-point rounding effects.",
    "",
    "## Field-response comparison",
    "",
    field_comparison.to_markdown(index=False),
    "",
    "## Reconstructed observed-acceleration comparison",
    "",
    gobs_comparison.to_markdown(index=False),
    "",
    "## Execution environment",
    "",
    f"- Python: {platform.python_version()}",
    f"- numpy: {version('numpy')}",
    f"- pandas: {version('pandas')}",
    f"- scikit-learn: {version('scikit-learn')}",
]

certificate_path = RUN_DIR / "VALIDATION_CERTIFICATE.md"

certificate_path.write_text(
    "\n".join(certificate_lines) + "\n",
    encoding="utf-8",
)

summary_payload = {
    "validation_status": "PASS" if overall_pass else "FAIL",
    "field_metrics_exact": field_pass,
    "gobs_metrics_exact": gobs_pass,
    "metric_tolerance": 1e-10,
    "n_frozen_outer_fold_fits": int(len(exact_fold_check)),
    "n_reconstructed_oof_rows": int(len(exact_oof)),
    "samples": sorted(field_comparison["sample"].unique().tolist()),
    "kernel_variants": sorted(
        field_comparison["kernel_variant"].unique().tolist()
    ),
}

summary_path = RUN_DIR / "validation_summary.json"

summary_path.write_text(
    json.dumps(summary_payload, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

checksum_lines = [
    "# SHA-256 manifest for this frozen-validation reproduction run"
]

run_files = sorted(
    path for path in RUN_DIR.rglob("*")
    if path.is_file() and path.name != "SHA256SUMS.txt"
)

for path in run_files:
    relative_name = path.relative_to(RUN_DIR).as_posix()
    checksum_lines.append(f"{sha256_file(path)}  {relative_name}")

checksum_path = RUN_DIR / "SHA256SUMS.txt"

checksum_path.write_text(
    "\n".join(checksum_lines) + "\n",
    encoding="utf-8",
)

print("=== FROZEN VALIDATION CERTIFIED ===")
print("Overall status:", "PASS" if overall_pass else "FAIL")
print("Field metrics exact:", field_pass)
print("g_obs metrics exact:", gobs_pass)
print("Frozen outer-fold fits:", len(exact_fold_check))
print("Reconstructed OOF rows:", len(exact_oof))
print("Certificate:", certificate_path)
print("Summary:", summary_path)
print("Checksums:", checksum_path)

if not overall_pass:
    raise RuntimeError("Frozen validation did not pass.")

=== FROZEN VALIDATION CERTIFIED ===
Overall status: PASS
Field metrics exact: True
g_obs metrics exact: True
Frozen outer-fold fits: 60
Reconstructed OOF rows: 30240
Certificate: /content/drive/MyDrive/UQSH_DarkMatter_Project/reproducibility_exports/sparc-clean-independent-reconstruction-20260812_124457_UTC/SPARC_Clean_Independent_Reconstruction/public_colab_runs/frozen_validation_20260813_080154_UTC/VALIDATION_CERTIFICATE.md
Summary: /content/drive/MyDrive/UQSH_DarkMatter_Project/reproducibility_exports/sparc-clean-independent-reconstruction-20260812_124457_UTC/SPARC_Clean_Independent_Reconstruction/public_colab_runs/frozen_validation_20260813_080154_UTC/validation_summary.json
Checksums: /content/drive/MyDrive/UQSH_DarkMatter_Project/reproducibility_exports/sparc-clean-independent-reconstruction-20260812_124457_UTC/SPARC_Clean_Independent_Reconstruction/public_colab_runs/frozen_validation_20260813_080154_UTC/SHA256SUMS.txt
